# Enterprise RAG — Hands-On, Part 4 of 11: Chunking and ingestion

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [1]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

project root : d:\INTERVIEW PREPARATION\DevRev_Preparation\enterprise_rag_platform
corpus       : data\corpus
api key      : found
embed model  : text-embedding-3-small
chat model   : gpt-4o-mini


### Setup — recap of state from earlier parts


In [2]:
from enterprise_rag.ingest.loader import load_corpus

docs = load_corpus()

---
# Part 4 - Chunking and ingestion

Chunks split on markdown headings first, then pack to a target size. Every chunk **inherits its
parent's ACL attributes** - that denormalisation is what makes a single-pass pre-filter possible.

Ingestion does two independent things with each validated document: it chunks and embeds it into the
vector index (below), **and** it writes one row per document into a separate ACL catalog (SQLite) -
see `ingest/catalog.py`. The catalog write does not depend on chunking or embedding at all, which is
the whole point: a later access-rule change only ever needs a write to that one row, never a
re-chunk or a re-embed.

In [3]:
from enterprise_rag.ingest.chunker import chunk_document

doc = next(d for d in docs if d.attrs.doc_id == "CT-VTX-001")
chunks = chunk_document(doc)
print(f"{doc.attrs.doc_id} -> {len(chunks)} chunks\n")
for c in chunks[:4]:
    print(f"[{c.chunk_id}] section={c.section!r}  ({len(c.text)} chars)")
    print(textwrap.indent(textwrap.fill(c.text[:200], 88), "    "), "\n")

CT-VTX-001 -> 5 chunks

[CT-VTX-001#0] section='Master Services Agreement - Vertex Financial Ltd'  (214 chars)
    Master Services Agreement - Vertex Financial - Master Services Agreement - Vertex
    Financial Ltd  **Effective:** 2025-07-01   **Term:** 36 months   **Region:** EU
    (Frankfurt) **Annual contract value:** 

[CT-VTX-001#1] section='Service commitments'  (260 chars)
    Master Services Agreement - Vertex Financial - Service commitments  - Monthly
    availability commitment: **99.9%** measured on successful ingest acceptance. -
    Contracted sustained ingest ceiling: **2,00 

[CT-VTX-001#2] section='Service credits'  (494 chars)
    Master Services Agreement - Vertex Financial - Service credits  If monthly availability
    falls below the commitment, Vertex is entitled to service credits against the following
    month's fees:  - Below 9 

[CT-VTX-001#3] section='Data residency'  (279 chars)
    Master Services Agreement - Vertex Financial - Data residency  All Vertex t

In [4]:
# The service-credit tiers survive as ONE coherent chunk - the whole point of
# structure-aware splitting.
credits = next(c for c in chunks if "credit" in c.section.lower())
print(credits.text)

Master Services Agreement - Vertex Financial - Service credits

If monthly availability falls below the commitment, Vertex is entitled to service credits against
the following month's fees:

- Below 99.9% but at or above 99.5%: **10%** credit
- Below 99.5% but at or above 99.0%: **25%** credit
- Below 99.0%: **50%** credit

Credits must be claimed by the customer in writing within 30 days of the end of the affected
month. Credits are the sole and exclusive remedy for availability failures.


In [5]:
# Every chunk carries the parent's permissions, and metadata is flattened to
# Chroma-compatible scalars (note the grp__* boolean columns).
print(json.dumps(credits.to_metadata(), indent=2))

{
  "chunk_id": "CT-VTX-001#2",
  "doc_id": "CT-VTX-001",
  "title": "Master Services Agreement - Vertex Financial",
  "section": "Service credits",
  "ordinal": 2,
  "tenant_id": "meridian",
  "source": "contract",
  "product": "platform",
  "owner": "legal",
  "sensitivity": "confidential",
  "sensitivity_level": 2,
  "region": "EU",
  "contains_pii": false,
  "need_to_know": "",
  "valid_from": "",
  "valid_until": "",
  "grp__sales": true,
  "grp__legal": true,
  "grp__account-management": true
}


In [6]:
# Build the index if it is not already there.
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
    print("index already built:", stats)
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

index already built: {'chunks': 86, 'documents': 22, 'by_source': {'contract': 12, 'helpcenter': 14, 'postmortem': 12, 'pricing': 9, 'runbook': 16, 'advisory': 9, 'ticket': 14}, 'by_sensitivity': {'confidential': 33, 'public': 14, 'internal': 30, 'restricted': 9}}


---

**◀ Previous:** [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb)

**Next ▶:** [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb)
